# Stochastic Calculus from Zero · Episode 19: Square-root diffusions: CIR

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> **The short rate** (illustrative): starts at 3.85%, pulled towards 3.50% with mean reversion 0.30 and volatility 1% a year (Vasicek). Made up for teaching. Here the volatility is 0.051 × √r, which is 1% at 3.85%.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The SDE · 4. Feller's condition · 5. Touching zero · 6. Same mean, different shape · 7. Bond prices · 8. Heston's variance · 9. This episode's question · 10. Exercises · 11. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 19,
    "quantlib_version": ql.__version__,
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))

## 2. Last episode's answer

Half-life $= \ln 2 / a = \ln 2 / 0.6 = 1.16$ years: double the pull, half the half-life.

In [ ]:
out["prev"] = {"answer": math.log(2) / 0.6}

## 3. The SDE

Cox, Ingersoll and Ross (1985) multiply the noise by $\sqrt{r}$:

$$dr_t = a(\theta - r_t)\,dt + \sigma\sqrt{r_t}\,dW_t .$$

Near zero the noise shrinks while the drift $a\theta$ stays positive, so the rate is pushed back up. We pick $\sigma$ so the volatility at the start, $\sigma\sqrt{r_0}$, matches Vasicek's 1%.

In [ ]:
R0, THETA, A = 0.0385, 0.035, 0.30
SIG = 0.01 / math.sqrt(R0)
SIG_BAD = 0.25
out["inputs"] = {"r0_pct": R0 * 100, "theta_pct": THETA * 100, "a": A, "sigma": SIG, "sigma_bad": SIG_BAD,
                 "vol_at_r0_pct": SIG * math.sqrt(R0) * 100, "year": 1985, "feller_year": 1951}
out["inputs"]

## 4. Feller's condition

Feller (1951) showed that a square-root diffusion never reaches zero if

$$2a\theta \ge \sigma^2 .$$

The push away from zero, $a\theta$, has to beat the noise. We test our $\sigma = 0.051$ and a much larger $\sigma = 0.25$.

In [ ]:
feller = {"lhs": 2 * A * THETA, "rhs_ok": SIG ** 2, "rhs_bad": SIG_BAD ** 2, "boundary_sigma": math.sqrt(2 * A * THETA)}
assert feller["rhs_ok"] < feller["lhs"] < feller["rhs_bad"]
out["feller"] = feller
feller

## 5. Touching zero

Simulate 20,000 paths for 5 years with daily steps. The square root needs $r \ge 0$, and a discrete step can overshoot below zero, so we use the **full-truncation** Euler scheme: inside the drift and the square root, a negative $r$ counts as 0.

In [ ]:
rng = np.random.default_rng(seed)

def cir_paths(sig, n_paths=20_000, years=5, per_year=252, keep=30):
    x = np.full(n_paths, R0); hit = np.zeros(n_paths, bool); dt = 1 / per_year
    kept = [x[:keep].copy()]
    for i in range(years * per_year):
        xp = np.maximum(x, 0)
        x = x + A * (THETA - xp) * dt + sig * np.sqrt(xp * dt) * rng.standard_normal(n_paths)
        hit |= x <= 0
        if (i + 1) % 5 == 0:
            kept.append(np.maximum(x[:keep], 0))
    return np.maximum(x, 0), hit, np.array(kept).T

x_ok, hit_ok, fan_ok = cir_paths(SIG)
x_bad, hit_bad, fan_bad = cir_paths(SIG_BAD)
tt = np.arange(fan_ok.shape[1]) * 5 / 252
out["zero"] = {"paths": len(x_ok), "hit_ok": float(hit_ok.mean()), "hit_bad": float(hit_bad.mean()),
               "hit_ok_pct": float(hit_ok.mean() * 100), "hit_bad_pct": float(hit_bad.mean() * 100),
               "t": tt.round(4).tolist(), "fan_ok": (fan_ok * 100).round(3).tolist(), "fan_bad": (fan_bad * 100).round(3).tolist(),
               "ticks": [{"x": x, "label": f"{x}Y" if x else "0"} for x in range(6)]}
assert hit_ok.mean() == 0 and hit_bad.mean() > 0.5
print(f"touched zero: {hit_ok.mean():.1%} with sigma {SIG:.3f}, {hit_bad.mean():.1%} with sigma {SIG_BAD}")

## 6. Same mean, different shape

Taking expectations kills the $dW$ term, so the mean solves the same equation as Vasicek's, $m' = a(\theta - m)$: the means are identical. The variance (Cox, Ingersoll and Ross give it in closed form) and the shape differ: CIR rates follow a scaled non-central chi-squared distribution, skewed to the right and never negative.

In [ ]:
m5, sd5 = cir_moments(5.0, A, THETA, SIG, R0)
vas_m5, vas_sd5 = ou_moments(5.0, A, THETA, 0.01, R0)
out["moments"] = {"mean_pct": m5 * 100, "sd_pct": sd5 * 100, "sim_mean_pct": float(x_ok.mean() * 100),
                  "sim_sd_pct": float(x_ok.std() * 100), "vasicek_mean_pct": vas_m5 * 100, "vasicek_sd_pct": vas_sd5 * 100,
                  "min_pct": float(x_ok.min() * 100)}
assert abs(m5 - vas_m5) < 1e-15
assert abs(x_ok.mean() / m5 - 1) < 0.01 and abs(x_ok.std() / sd5 - 1) < 0.03
pd.DataFrame([out["moments"]])

## 7. Bond prices

A zero-coupon bond paying 1 in 5 years is worth $E[e^{-\int_0^5 r\,dt}]$ under the pricing measure (Episodes 21 and 23 explain why). CIR has a closed form, and so does QuantLib's `CoxIngersollRoss` model.

In [ ]:
bond = {"closed": float(cir_bond(5.0, R0, A, THETA, SIG)),
        "ql": ql.CoxIngersollRoss(R0, THETA, A, SIG).discountBond(0.0, 5.0, R0),
        "vasicek": float(vasicek_bond(5.0, R0, A, THETA, 0.01))}
assert abs(bond["closed"] - bond["ql"]) < 1e-10
bond["diff_bp_price"] = (bond["closed"] - bond["vasicek"]) * 1e4
out["bond"] = bond
bond

## 8. Heston's variance

Series 3's Heston model (s3-ep29) lets the share's variance follow exactly this SDE, $dv = \kappa(\bar v - v)\,dt + \xi\sqrt{v}\,dW$. Feller's condition $2\kappa\bar v \ge \xi^2$ decides whether the variance can hit zero, and calibrated Heston models often break it: that is why Heston simulations need schemes like full truncation.

## 9. This episode's question

With $\sigma = 0.1$: $\sigma^2 = 0.01 < 2a\theta = 0.021$, so Feller's condition holds and the rate never reaches zero.

In [ ]:
out["question"] = {"sigma": 0.1, "rhs": 0.01, "lhs": 2 * A * THETA, "met": 0.01 < 2 * A * THETA}
assert out["question"]["met"]

## 10. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Is Feller's condition met for $a = 0.5$, $\theta = 4\%$, $\sigma = 0.15$? And for $\sigma = 0.25$?

<details><summary>Answer</summary>

$2a\theta = 0.04$. $\sigma^2 = 0.0225 \le 0.04$: met. $\sigma^2 = 0.0625 > 0.04$: not met, zero can be reached.

</details>

**Exercise 2 (By hand).** Show that the CIR mean $m(t) = E[r_t]$ satisfies $m' = a(\theta - m)$, the same equation as Vasicek's.

<details><summary>Answer</summary>

Integrate the SDE and take expectations: $E[\int \sigma\sqrt{r}\,dW] = 0$ (an Itô integral is a martingale), so $m(t) = r_0 + \int_0^t a(\theta - m(s))\,ds$. Differentiate.

</details>

**Exercise 3 (By hand).** Apply Itô to $X = \sqrt{r}$. What is the volatility of $X$?

<details><summary>Answer</summary>

$f' = \tfrac{1}{2\sqrt r}$, $f'' = -\tfrac{1}{4} r^{-3/2}$, $(dr)^2 = \sigma^2 r\,dt$: $dX = \big(\tfrac{a\theta - \sigma^2/4}{2X} - \tfrac{a}{2}X\big)dt + \tfrac{\sigma}{2}\,dW$. The square root has constant volatility $\sigma/2$; the drift explodes upwards near 0 when $a\theta > \sigma^2/4$.

</details>

**Exercise 4 (In code).** Find the fraction of paths that touch zero within 5 years for $\sigma$ from 0.05 to 0.25. Where does it start rising?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
rows = []
for s in (0.05, 0.10, 0.145, 0.20, 0.25):
    _, h, _ = cir_paths(s, n_paths=5_000)
    rows.append({"sigma": s, "sigma_sq": s * s, "2a theta": 2 * A * THETA, "touched zero": float(h.mean())})
assert rows[0]["touched zero"] == 0 and rows[-1]["touched zero"] > 0.5
print(pd.DataFrame(rows).to_string(index=False))
print("Feller boundary: sigma =", round(math.sqrt(2 * A * THETA), 3), "(daily steps blur it a little)")

**Exercise 5 (In code).** Price the 5-year CIR bond by Monte Carlo: average $e^{-\int_0^5 r\,dt}$ over simulated paths, and compare with the closed form.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
rng_b = np.random.default_rng(3)
n, dt = 20_000, 1 / 252
x = np.full(n, R0); integral = np.zeros(n)
for _ in range(5 * 252):
    xp = np.maximum(x, 0)
    xn = x + A * (THETA - xp) * dt + SIG * np.sqrt(xp * dt) * rng_b.standard_normal(n)
    integral += 0.5 * (np.maximum(x, 0) + np.maximum(xn, 0)) * dt
    x = xn
mc = np.exp(-integral)
se = mc.std() / math.sqrt(n)
assert abs(mc.mean() - bond["closed"]) < 3 * se + 1e-4
print(f"Monte Carlo {mc.mean():.5f} ± {se:.5f}; closed form {bond['closed']:.5f}")

## 11. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")